In [1]:
import random
from datasets import DatasetDict, load_dataset
from huggingface_hub import whoami


username = whoami()["name"]
dataset = load_dataset("rajpurkar/squad_v2", "squad_v2")
# prompt format from [How to Train Long-Context Language Models (Effectively)](https://aclanthology.org/2025.acl-long.366/) (Gao et al., ACL 2025)

refusal_instruction = "If the context does not contain the answer, reply with: 'The question cannot be answered from this context.'"
prompts = [
    "Given the document, please answer the question.",
    "Here is a piece of text; answer the following question based on it.",
    "Please answer the question using the provided content.",
    "Based on the given passage, respond to the question.",
    "Read the snippet and answer the question that follows.",
    "Using the provided text, answer the following question."
]

templates = [
    "{prompt} {refusal_instruction}\n\n{documents}\n\nQuestion: {question}",
    "{prompt} {refusal_instruction}\n\n==== document starts ====\n{documents}\n==== document ends ====\n\nQuestion:\n{question}",
    "{prompt} {refusal_instruction}\n\n{documents}\n\n{question}",
    "{prompt} {refusal_instruction} Question: {question}\n\n{documents}",
    "{prompt} {refusal_instruction} {question}\n\n{documents}",
    "{prompt} {refusal_instruction} \n\n{question}\n\n{documents}",
]

def convert(example, idx=None):
    prompt = prompts[idx % len(prompts)]
    template = templates[(idx // len(prompts)) % len(templates)]

    user_message = template.format(
        prompt=prompt,
        documents=example["context"],
        question=example["question"],
        refusal_instruction=refusal_instruction
    )

    answer_text = (
        example["answers"]["text"][0]
        if len(example["answers"]["text"]) > 0
        else "The question cannot be answered from this context."
    )

    return {
        "id": example["id"],
        "title": example["title"],
        "context": example["context"],
        "question": example["question"],
        "answers": example["answers"],
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer_text},
        ],
        "source": "rajpurkar/squad_v2",
        "prompt_used": prompt,
    }

train_ds = dataset["train"].shuffle(seed=42).map(
    convert,
    with_indices=True,
    num_proc=1,
    remove_columns=dataset["train"].column_names
)

validation_ds = dataset["validation"].shuffle(seed=42).map(
    convert,
    with_indices=True,
    num_proc=1,
    remove_columns=dataset["validation"].column_names
)

conversational_dataset = DatasetDict({
    "train": train_ds.shuffle(seed=42),
    "validation": validation_ds.shuffle(seed=42)
})

conversational_dataset.push_to_hub(
    "squad_v2_conversational",
    private=True
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/131 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/loris3/squad_v2_conversational/commit/71440d51998833d4f46beed09c0be94310d485d0', commit_message='Upload dataset', commit_description='', oid='71440d51998833d4f46beed09c0be94310d485d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/loris3/squad_v2_conversational', endpoint='https://huggingface.co', repo_type='dataset', repo_id='loris3/squad_v2_conversational'), pr_revision=None, pr_num=None)

In [2]:
validation_dataset[0]

{'id': '56ddde6b9a695914005b9628',
 'title': 'Normans',
 'context': 'The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of the 10th century, and it continued to evolve over the succeeding centuries.',
 'question': 'In what country is Normandy located?',
 'answers': {'text': ['France', 'France', 'France', 'France'],
  'answer_start': [159, 159, 159, 159]},
 'messages': [{'content': 'Gi

In [3]:
validation_dataset[10]

{'id': '56dddf4066d3e219004dad60',
 'title': 'Normans',
 'context': 'The Norman dynasty had a major political, cultural and military impact on medieval Europe and even the Near East. The Normans were famed for their martial spirit and eventually for their Christian piety, becoming exponents of the Catholic orthodoxy into which they assimilated. They adopted the Gallo-Romance language of the Frankish land they settled, their dialect becoming known as Norman, Normaund or Norman French, an important literary language. The Duchy of Normandy, which they formed by treaty with the French crown, was a great fief of medieval France, and under Richard I of Normandy was forged into a cohesive and formidable principality in feudal tenure. The Normans are noted both for their culture, such as their unique Romanesque architecture and musical traditions, and for their significant military accomplishments and innovations. Norman adventurers founded the Kingdom of Sicily under Roger II after conquering

In [ ]:
from datasets import load_dataset
from huggingface_hub import whoami
username = whoami()["name"]

dataset = load_dataset("DataAttributionEval/ftrace", "Llama-3.1-8B")
def convert(example):
    return {
        "id": example["data_id"],
        "messages": [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["response"]},
        ],
        "source": "DataAttributionEval/ftrace",
        "facts": example["facts"]
    }

conversational_dataset = dataset["train"].map(
    convert,
    remove_columns=dataset["train"].column_names
)

conversational_dataset.push_to_hub(
    "ftrace_date-lm_conversational",
    split="train",
    private=True
)

Dataset({
    features: ['data_id', 'prompt', 'response', 'facts'],
    num_rows: 6708
})

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/446 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/loris3/ftrace_date-lm_conversational/commit/130d0e2dd87962ffd21a00f74685425fb50ddf91', commit_message='Upload dataset', commit_description='', oid='130d0e2dd87962ffd21a00f74685425fb50ddf91', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/loris3/ftrace_date-lm_conversational', endpoint='https://huggingface.co', repo_type='dataset', repo_id='loris3/ftrace_date-lm_conversational'), pr_revision=None, pr_num=None)